# Lab 1: NumPy for Cat and Dog Faces

In this notebook, you will treat **cat and dog face images** as NumPy arrays and build a small hand-crafted feature matrix.

This version focuses on core NumPy image operations and keeps the workflow concrete:

- load an image into a NumPy array
- crop and flip with slicing
- normalize to `[0, 1]`
- convert RGB to grayscale
- compute summaries with `axis=`
- apply a small filter with a kernel and matrix multiplication
- flatten an image into one vector
- engineer features with `np.concatenate(...)` and `np.apply_along_axis(...)`
- stack features into a feature matrix for later machine learning work

**Dataset assumption**

Use the curated cat-and-dog-faces dataset extracted into:

`data/`

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from lab_utils.visualization import (
    plot_feature_vector,
    show_image_gallery,
)

# Safe project root (works in scripts + notebooks)
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"

LABELS = ("cat", "dog")
LABEL_TO_INDEX = {"cat": 0, "dog": 1}

IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

SEED = 1234

def label_from_path(path: Path) -> str:
    label = path.parent.name
    if label not in LABEL_TO_INDEX:
        raise ValueError(f"Unexpected label folder: {path}")
    return label


def load_preview_image(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"))


def list_image_paths(label: str) -> list[Path]:
    label_dir = DATA_ROOT / label
    paths = []
    for pattern in IMAGE_EXTENSIONS:
        paths.extend(label_dir.glob(pattern))
    return sorted(paths)

def shuffled_paths(paths: list[Path], seed_offset: int = 0) -> list[Path]:
    rng = np.random.default_rng(SEED + seed_offset)
    indices = rng.permutation(len(paths))
    return [paths[int(idx)] for idx in indices]

def sample_paths(paths: list[Path], count: int, seed_offset: int) -> list[Path]:
    ordered = shuffled_paths(paths, seed_offset=seed_offset)
    return ordered[: min(count, len(ordered))]


def sample_per_class(paths: list[Path], n_per_class: int, seed_offset: int = 0) -> list[Path]:
    sampled = []
    for label_index, label in enumerate(LABELS):
        label_paths = [path for path in paths if label_from_path(path) == label]
        sampled.extend(sample_paths(label_paths, n_per_class, seed_offset + 50 * label_index))
    return sampled

def split_train_test(paths: list[Path], train_ratio: float = 0.7, seed_offset: int = 0):
    shuffled = shuffled_paths(paths, seed_offset)
    split_idx = int(len(shuffled) * train_ratio)
    return shuffled[:split_idx], shuffled[split_idx:]


# Check dataset exists
expected = [
    DATA_ROOT / "cat",
    DATA_ROOT / "dog",
]
if not all(path.exists() for path in expected):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_ROOT}. Expected 'cat' and 'dog' folders."
    )


# Load all paths
cat_paths = list_image_paths("cat")
dog_paths = list_image_paths("dog")
cat_dog_paths = cat_paths + dog_paths

# Split per class (7:3)
cat_train, cat_test = split_train_test(cat_paths, 0.7, seed_offset=0)
dog_train, dog_test = split_train_test(dog_paths, 0.7, seed_offset=100)

# Combine
train_paths = cat_train + dog_train
test_paths = cat_test + dog_test

print(f"Using dataset from: {DATA_ROOT}")
print(f"Found {len(cat_paths)} cat images")
print(f"Found {len(dog_paths)} dog images")

if len(cat_paths) == 0 or len(dog_paths) == 0:
    raise ValueError("No images found. Check folder paths or file extensions.")

### Visual Helper: Preview the Faces Dataset

Before starting the TODOs, look at a few cat and dog face images from the student-specific subset.


In [ ]:
preview_paths = sample_per_class(cat_dog_paths, n_per_class=3, seed_offset=10)
preview_images = [load_preview_image(path) for path in preview_paths]
preview_titles = [f"{label_from_path(path)}: {path.name}" for path in preview_paths]
show_image_gallery(
    preview_images,
    titles=preview_titles,
    ncols=3,
    figsize=(10, 6),
    suptitle="Cat and dog face preview",
)
plt.show()


## Question 1: Load one image into a NumPy array

Write a function that:

- opens one file from disk
- converts it to RGB
- returns an `H x W x C` NumPy array

This is the starting point for every later NumPy operation in the lab.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def get_image_data(file_path: Path) -> np.ndarray:
    img_obj = Image.open(file_path).convert("RGB")
    return np.asarray(img_obj)

test_path = cat_paths[0]
img_array = get_image_data(test_path)

print(f"Structure: {img_array.shape}")
print(f"Data Type: {img_array.dtype}")
print(f"Min/Max Value: {img_array.min()} - {img_array.max()}")

show_image_gallery(
    [img_array], 
    titles=[test_path.name], 
    ncols=1, 
    figsize=(5, 5)
)
plt.show()

## Question 2: Crop the image with slicing

Implement a centered square crop. Keep the crop size at `48 x 48` for the rest of the lab so the crop is visible and later operations stay consistent.


In [ ]:
def center_crop(img: np.ndarray, size: int = 48) -> np.ndarray:
    h, w = img.shape[:2]
    
    start_y = (h - size) // 2
    start_x = (w - size) // 2
    
    return img[start_y : start_y + size, start_x : start_x + size]

img_cropped = center_crop(sample_image, 48)

print(f"New dimensions: {img_cropped.shape}")

show_image_gallery(
    [sample_image, img_cropped],
    titles=["Source Image", "Cropped Result"],
    ncols=2,
    figsize=(9, 4)
)
plt.show()

## Question 3: Flip the crop horizontally

Mirror the cropped image from left to right using slicing only.


In [ ]:
def flip_horizontal(img: np.ndarray) -> np.ndarray:
    return img[:, ::-1, ...]

horizontal_img = flip_horizontal(cropped_image)

show_image_gallery(
    [cropped_image, horizontal_img],
    titles=["Input Crop", "Flipped View"],
    ncols=2,
    figsize=(7, 3.5)
)
plt.show()

## Question 4: Normalize pixels to `[0, 1]`

Convert the cropped RGB image from unsigned integers into `float32` values in the range `[0, 1]`.


In [ ]:
def normalize_01(image: np.ndarray) -> np.ndarray:
    # TODO: convert the image to float32 and divide by 255.
    raise NotImplementedError("Normalize pixel values to [0, 1].")


def show_histograms(uint8_img, float_img):
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.hist(uint8_img.ravel(), bins=50)
    plt.title("Before (uint8: 0–255)")

    plt.subplot(1, 2, 2)
    plt.hist(float_img.ravel(), bins=50)
    plt.title("After (float: 0–1)")

    plt.tight_layout()
    plt.show()

sample_float = normalize_01(cropped_image)

# 1. Side-by-side image (Both image will look the same)
show_image_gallery(
    [cropped_image, sample_float],
    titles=["uint8 (0–255)", "float (0–1)"],
    ncols=2,
    figsize=(8, 4),
)

# 2. Stats
print("Before:", cropped_image.dtype, cropped_image.min(), cropped_image.max())
print("After :", sample_float.dtype, sample_float.min(), sample_float.max())

# 3. Histogram
show_histograms(cropped_image, sample_float)

plt.show()

## Question 5: Convert RGB to grayscale

Turn the normalized RGB image into a single grayscale array using standard RGB weights 

$GREY = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$.


In [ ]:
def rgb_to_gray(img_rgb: np.ndarray) -> np.ndarray:
    # Sử dụng trọng số chuẩn cho các kênh R, G, B
    weights = np.array([0.2989, 0.5870, 0.1140], dtype=np.float32)
    return np.dot(img_rgb[..., :3], weights)

gray_img = rgb_to_gray(sample_float)

print(f"Grayscale shape: {gray_img.shape}")
print(f"Data type: {gray_img.dtype}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

ax1.imshow(sample_float)
ax1.set_title("Input RGB")
ax1.axis("off")

ax2.imshow(gray_img, cmap="gray")
ax2.set_title("Resulting Gray")
ax2.axis("off")

plt.tight_layout()
plt.show()

## Question 6: Use `axis=` to summarize channels

Compute one mean value per color channel with `axis=(0, 1)`, then choose the brightest channel with `np.argmax(...)`.


In [ ]:
CHANNEL_NAMES = np.array(["red", "green", "blue"])

def channel_summary(img_float: np.ndarray) -> tuple[np.ndarray, int]:
    # Tính giá trị trung bình trên các trục không gian (H, W)
    avg_values = img_float.mean(axis=(0, 1))
    # Tìm vị trí của giá trị lớn nhất
    top_channel_idx = np.argmax(avg_values)
    
    return avg_values, top_channel_idx

means, brightest_idx = channel_summary(sample_float)

print(f"Mean per channel: {means}")
print(f"Peak intensity: {CHANNEL_NAMES[brightest_idx]}")

# Thiết lập biểu đồ với phong cách khác biệt
fig, ax = plt.subplots(figsize=(6, 3.5))
bar_colors = ["tab:red", "tab:green", "tab:blue"]

ax.bar(CHANNEL_NAMES, means, color=bar_colors, edgecolor="black", alpha=0.8)
ax.set_title("Mean Intensity Analysis")
ax.set_ylabel("Brightness Score")
ax.set_ylim(0, 1)  # Cố định trục y từ 0 đến 1 vì ảnh đã normalize

plt.tight_layout()
plt.show()

## Question 7: Apply a filter with a kernel and matrix multiplication

Implement a tiny 2D convolution on the grayscale image. At each location:

1. take a `3 x 3` patch
2. flatten the patch and kernel
3. multiply them with `@`

Use the Laplacian kernel from the setup cell.


In [ ]:
EDGE_KERNEL = np.array(
    [[0, 1, 0], 
     [1, -4, 1], 
     [0, 1, 0]], 
    dtype=np.float32
)

def convolve2d_matmul(img_gray: np.ndarray, filter_kernel: np.ndarray) -> np.ndarray:
    ih, iw = img_gray.shape
    kh, kw = filter_kernel.shape
    
    # Tính toán kích thước đầu ra (vùng hợp lệ)
    out_h = ih - kh + 1
    out_w = iw - kw + 1
    output = np.zeros((out_h, out_w))
    
    # Làm phẳng kernel để chuẩn bị cho tích vô hướng
    flat_kernel = filter_kernel.flatten()
    
    for i in range(out_h):
        for j in range(out_w):
            # Trích xuất vùng ảnh (patch) và làm phẳng
            patch = img_gray[i : i + kh, j : j + kw].flatten()
            # Thực hiện nhân ma trận/tích vô hướng
            output[i, j] = patch @ flat_kernel
            
    return output

# Thực thi và hiển thị kết quả
filtered_res = convolve2d_matmul(sample_gray, EDGE_KERNEL)

print(f"Output dimensions: {filtered_res.shape}")

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(9, 4))

ax_left.imshow(sample_gray, cmap="gray")
ax_left.set_title("Input Gray")
ax_left.axis("off")

# Hiển thị giá trị tuyệt đối để làm nổi bật các cạnh
ax_right.imshow(np.abs(filtered_res), cmap="inferno")
ax_right.set_title("Edge Detection")
ax_right.axis("off")

plt.tight_layout()
plt.show()

## Question 8: Flatten one image into one vector

Take the grayscale crop and turn it into a one-dimensional vector.


In [ ]:
def flatten_image(image: np.ndarray) -> np.ndarray:
    # TODO: return a 1D vector version of the image.
    raise NotImplementedError("Flatten the image into one vector.")


sample_flat = flatten_image(sample_gray)
print("original shape:", sample_gray.shape)
print("flat shape:", sample_flat.shape)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(sample_flat[:256], color="#4C6FFF")
ax.set_title("First 256 grayscale values after flattening")
ax.set_xlabel("Index")
ax.set_ylabel("Value")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Question 9: Engineer a feature vector with `concatenate` and `apply`

Build one hand-crafted feature vector that combines:

- RGB means
- RGB standard deviations
- the brightest channel index
- the mean and standard deviation of the filtered response
- one summary from `np.apply_along_axis(...)`

Use `np.concatenate(...)` to join the pieces.


In [ ]:
FEATURE_NAMES = [
    "mean_r",
    "mean_g",
    "mean_b",
    "std_r",
    "std_g",
    "std_b",
    "brightest_channel",
    "edge_mean",
    "edge_std",
    "row_std_mean",
]


def extract_features(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    cropped = center_crop(image, crop_size=48)
    image_float = normalize_01(cropped)
    gray = rgb_to_gray(image_float)
    channel_means, brightest_channel = channel_summary(image_float)
    channel_stds = image_float.std(axis=(0, 1)).astype(np.float32)
    filtered = convolve2d_matmul(gray, kernel)
    row_std_profile = np.apply_along_axis(np.std, 1, gray)

    # TODO: use np.concatenate to combine the feature pieces into one float32 vector.
    raise NotImplementedError("Build one hand-crafted feature vector.")


sample_features = extract_features(sample_image, EDGE_KERNEL)
print("feature shape:", sample_features.shape)
fig, ax = plot_feature_vector(sample_features, FEATURE_NAMES, title="Sample NumPy feature vector")
plt.show()

## Question 10: Build and inspect a feature matrix

Apply your feature function to the small balanced train/test subsets from the face dataset.

Tasks:

1. build one feature matrix for the train images and one for the test images
2. return the matching integer labels
3. print the resulting shapes
4. compute an overall feature mean with `axis=0`
5. visualize the feature matrix and the average feature vector


In [ ]:
def build_feature_matrix(image_paths: list[Path], filter_kernel: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    data_list = []
    labels = []
    
    for p in image_paths:
        # Trích xuất đặc trưng (giả định hàm extract_features đã được bạn định nghĩa trước đó)
        f_vector = extract_features(p, filter_kernel)
        data_list.append(f_vector)
        
        # Gán nhãn dựa trên tên thư mục hoặc tên file (ví dụ: 'cat' -> 0, 'dog' -> 1)
        label_val = 1 if "dog" in p.parent.name.lower() else 0
        labels.append(label_val)
        
    return np.array(data_list), np.array(labels)

# Khởi tạo tập dữ liệu
X_train, y_train = build_feature_matrix(train_paths, EDGE_KERNEL)
X_test, y_test = build_feature_matrix(test_paths, EDGE_KERNEL)

# Hiển thị thống kê dữ liệu
print(f"Training set: {X_train.shape}, Labels: {y_train.shape}")
print(f"Testing set:  {X_test.shape}, Labels: {y_test.shape}")
print(f"Distribution (Train): {np.bincount(y_train)}")
print(f"Distribution (Test) : {np.bincount(y_test)}")

# Tính toán giá trị trung bình
avg_train_features = np.mean(X_train, axis=0)

# Trực quan hóa ma trận đặc trưng
fig, ax = plt.subplots(figsize=(11, 4))
img_plot = ax.imshow(X_train, aspect="auto", cmap="plasma")

ax.set_title("Feature Matrix Visualization (Training)")
ax.set_xlabel("Features")
ax.set_ylabel("Sample Index")

# Cài đặt nhãn trục hoành
ax.set_xticks(np.arange(len(FEATURE_NAMES)))
ax.set_xticklabels(FEATURE_NAMES, rotation=40, ha="right")

# Thêm thanh màu sắc (Colorbar)
plt.colorbar(img_plot, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

# Vẽ vector đặc trưng trung bình
plot_feature_vector(avg_train_features, FEATURE_NAMES, title="Mean Feature Representation")
plt.show()

## Reflection

Answer these short questions in your own words:

1. Why is it useful to keep the crop size fixed before feature extraction?
2. What does `axis=(0, 1)` mean when you compute channel means on an image?
3. What information does the small edge filter capture that plain RGB means miss?
4. Why does flattening help some operations but also lose spatial structure?
